In [1]:
import time
from math import comb
from pathlib import Path

import numpy as np
import pandas as pd
import pyscf, pyscf.cc, pyscf.mcscf
import ffsim
from scipy.stats import spearmanr, kendalltau

# System (must match the ordering experiment exactly)
BOND     = 1.55
BASIS    = "6-31g"
N_FROZEN = 4
NORB     = 10
NELEC    = (3, 3)
BUDGET   = 15
SEED     = 2026

OUT = Path.home() / "sqd-project" / "outputs"
rng = np.random.default_rng(SEED)

DIM_A = comb(NORB, NELEC[0])
DIM_B = comb(NORB, NELEC[1])

print(f"N2 CAS({sum(NELEC)},{NORB}) @ {BOND} A")
print(f"alpha strings: {DIM_A}, beta strings: {DIM_B}, CI dim: {DIM_A*DIM_B}")

N2 CAS(6,10) @ 1.55 A
alpha strings: 120, beta strings: 120, CI dim: 14400


In [2]:
mol = pyscf.gto.Mole()
mol.build(atom=[["N", (0, 0, 0)], ["N", (0, 0, BOND)]],
          basis=BASIS, symmetry=False, verbose=0)
mf = pyscf.scf.RHF(mol).run(verbose=0)

active = range(N_FROZEN, N_FROZEN + NORB)
mol_data = ffsim.MolecularData.from_scf(mf, active_space=active)

cc = pyscf.cc.CCSD(mf,
        frozen=[i for i in range(mol.nao_nr()) if i not in active]).run(verbose=0)

cas = pyscf.mcscf.CASCI(mf, ncas=NORB, nelecas=NELEC)
cas.ncore = N_FROZEN
cas.run(verbose=0)

E_HF, E_CASCI = mol_data.hf_energy, cas.e_tot
ham = ffsim.linear_operator(mol_data.hamiltonian, norb=NORB, nelec=NELEC)
ref = ffsim.hartree_fock_state(NORB, NELEC)

print(f"HF    = {E_HF:.8f}")
print(f"CCSD  = {cc.e_tot:.8f}")
print(f"CASCI = {E_CASCI:.8f}")

HF    = -108.58732256
CCSD  = -108.82807465
CASCI = -108.84430043


In [3]:
op_full = ffsim.UCJOpSpinBalanced.from_t_amplitudes(t2=cc.t2, t1=cc.t1, n_reps=None)

alpha_alpha = [(p, p + 1) for p in range(NORB - 1)]
alpha_beta  = [(p, p) for p in range(0, NORB, 4)]

_m_aa = np.zeros((NORB, NORB), bool)
for p, q in alpha_alpha:
    _m_aa[p, q] = _m_aa[q, p] = True
np.fill_diagonal(_m_aa, True)

_m_ab = np.zeros((NORB, NORB), bool)
for p, q in alpha_beta:
    _m_ab[p, q] = _m_ab[q, p] = True

def permute_operator(op, perm):
    """Relabel orbital indices; permutation absorbed into the rotations.
    The UNMASKED operator is physically unchanged by this."""
    P = np.eye(op.norb)[list(perm)]
    J = np.asarray(op.diag_coulomb_mats)
    U = np.asarray(op.orbital_rotations)
    kw = dict(diag_coulomb_mats=np.einsum('ij,rsjk,lk->rsil', P, J, P),
              orbital_rotations=np.einsum('rij,kj->rik', U, P))
    if op.final_orbital_rotation is not None:
        kw['final_orbital_rotation'] = op.final_orbital_rotation
    return ffsim.UCJOpSpinBalanced(**kw)

def apply_mask(op):
    """Zero Jastrow elements outside the heavy-hex locality constraint."""
    J = np.asarray(op.diag_coulomb_mats).copy()
    J[:, 0] *= _m_aa
    J[:, 1] *= _m_ab
    kw = dict(diag_coulomb_mats=J,
              orbital_rotations=np.asarray(op.orbital_rotations))
    if op.final_orbital_rotation is not None:
        kw['final_orbital_rotation'] = op.final_orbital_rotation
    return ffsim.UCJOpSpinBalanced(**kw)

print(f"n_reps = {op_full.n_reps}")
print(f"same-spin pairs kept: {len(alpha_alpha)} / {NORB*(NORB-1)//2}")
print(f"opp-spin  pairs kept: {len(alpha_beta)} at p = {[p for p,_ in alpha_beta]}")

n_reps = 42
same-spin pairs kept: 9 / 45
opp-spin  pairs kept: 3 at p = [0, 4, 8]


In [4]:
def variational_energy(op):
    """<psi|H|psi>. Real/imag split because PySCF's contraction needs float64."""
    s = ffsim.apply_unitary(ref, op, norb=NORB, nelec=NELEC)
    sr = np.ascontiguousarray(s.real, dtype=np.float64)
    si = np.ascontiguousarray(s.imag, dtype=np.float64)
    return float(np.dot(sr, ham @ sr) + np.dot(si, ham @ si))

e0 = variational_energy(op_full)
devs = [abs(variational_energy(permute_operator(op_full, rng.permutation(NORB))) - e0)
        for _ in range(5)]
print(f"max deviation = {max(devs):.2e}")
assert max(devs) < 1e-8, "Permutation is not a symmetry -- index convention wrong"
print("GATE PASSED")

max deviation = 3.30e-12
GATE PASSED


In [5]:
def marginals_exact(op):
    """Exact infinite-shot alpha/beta marginals from |psi|^2.
    ffsim/pyscf store the CI vector as (n_alpha_strings, n_beta_strings)."""
    psi = ffsim.apply_unitary(ref, op, norb=NORB, nelec=NELEC)
    p = (np.abs(psi) ** 2).reshape(DIM_A, DIM_B)
    p = p / p.sum()
    return p.sum(axis=1), p.sum(axis=0)

def subspace_energy_exact(marg_a, marg_b, budget=BUDGET):
    """Diagonalise H in the product space of the top-`budget` strings."""
    ia = np.argsort(marg_a)[::-1][:budget]
    ib = np.argsort(marg_b)[::-1][:budget]
    idx = np.array([a * DIM_B + b for a in ia for b in ib])

    n = len(idx)
    basis = np.zeros((DIM_A * DIM_B, n))
    basis[idx, np.arange(n)] = 1.0

    Hb = np.column_stack([ham @ basis[:, k] for k in range(n)])
    Hs = basis.T @ Hb
    return float(np.linalg.eigvalsh((Hs + Hs.T) / 2)[0])

def gini(p):
    """Concentration of a distribution, ascending sort, range [0,1]."""
    p = np.sort(np.asarray(p, dtype=float))
    p = p / p.sum()
    n = len(p)
    return 1.0 - 2.0 * np.sum(p * (n - np.arange(1, n + 1) + 0.5) / n)

def evaluate_ordering(perm, budget=BUDGET):
    op = apply_mask(permute_operator(op_full, perm))
    ma, mb = marginals_exact(op)
    E = subspace_energy_exact(ma, mb, budget)
    return {
        "E_sub":        E,
        "err_sub_mHa":  (E - E_CASCI) * 1000,
        "gini_alpha":   gini(ma),
        "gini_beta":    gini(mb),
        "top1_alpha":   float(ma.max()),
        "top1_beta":    float(mb.max()),
        "neff_alpha":   float(1.0 / np.sum(ma ** 2)),
        "neff_beta":    float(1.0 / np.sum(mb ** 2)),
        "E_var":        variational_energy(op),
    }

t = time.time()
r = evaluate_ordering(np.arange(NORB))
dt = time.time() - t
print(f"One ordering: {dt:.3f} s  ({1/dt:.1f}/sec, {3600/dt:.0f}/hour)")
print()
for k, v in r.items():
    print(f"  {k:14s} {v:.6f}")

One ordering: 0.334 s  (3.0/sec, 10794/hour)

  E_sub          -108.805074
  err_sub_mHa    39.226513
  gini_alpha     0.990615
  gini_beta      0.990615
  top1_alpha     0.985722
  top1_beta      0.985722
  neff_alpha     1.029150
  neff_beta      1.029150
  E_var          -108.642444


In [6]:
rep = pd.read_csv(OUT / "seed_replication_n2_cas610_155.csv")
sbd_mean = rep.groupby("ordering")["err_sub_mHa"].mean()
print(f"Loaded {len(rep)} rows, {len(sbd_mean)} orderings\n")

# Rebuild the same permutations: identity + 25 draws from seed 2026
rng2 = np.random.default_rng(2026)
orderings = {"identity": np.arange(NORB)}
seen = {tuple(range(NORB))}
while len(orderings) <= 25:
    p = rng2.permutation(NORB)
    if tuple(p) not in seen:
        seen.add(tuple(p))
        orderings[f"p{len(orderings):02d}"] = p

rows = []
for name, perm in orderings.items():
    if name not in sbd_mean.index:
        continue
    rows.append({"ordering": name,
                 "fast_mHa": evaluate_ordering(perm)["err_sub_mHa"],
                 "sbd_mHa":  sbd_mean[name]})
fast = pd.DataFrame(rows).set_index("ordering")
print(f"Matched {len(fast)} orderings\n")

rho = spearmanr(fast.fast_mHa, fast.sbd_mHa).statistic
tau = kendalltau(fast.fast_mHa, fast.sbd_mHa).statistic
print(f"Spearman rho = {rho:+.3f}")
print(f"Kendall  tau = {tau:+.3f}")
print(f"mean |diff|  = {(fast.fast_mHa - fast.sbd_mHa).abs().mean():.2f} mHa\n")
print(fast.sort_values("sbd_mHa").round(2).to_string())

Loaded 130 rows, 26 orderings

Matched 26 orderings

Spearman rho = -0.083
Kendall  tau = -0.080
mean |diff|  = 20.19 mHa

          fast_mHa  sbd_mHa
ordering                   
p13          76.07    19.98
p07          86.64    20.26
p22          51.69    22.26
p05          48.99    23.71
p20          51.38    25.11
p12          51.47    25.58
p02          28.60    26.00
identity     39.23    26.02
p16          46.14    27.11
p18          47.06    27.75
p19         102.01    29.35
p01          41.02    38.48
p24          43.19    41.07
p11          53.32    41.60
p04          51.92    41.91
p08          51.69    43.16
p10          51.92    43.76
p21          34.71    44.11
p17          51.69    47.49
p09          79.33    49.62
p03          51.92    49.67
p15          51.69    50.08
p06          52.45    52.98
p25          38.89    53.40
p14          51.38    54.65
p23          36.35    96.58


In [7]:
# --- Is the reshape convention right? Test on the bare HF state. ---
psi_hf = ref
p_hf = (np.abs(psi_hf) ** 2).reshape(DIM_A, DIM_B)
print("HF state: number of nonzero entries =", np.count_nonzero(p_hf))
print("HF state: argmax (a_idx, b_idx) =", np.unravel_index(p_hf.argmax(), p_hf.shape))
print("expected (0, 0) for the HF determinant\n")

# --- Does the identity-ordering masked state look sane? ---
op_id = apply_mask(permute_operator(op_full, np.arange(NORB)))
ma, mb = marginals_exact(op_id)
print(f"identity: marg_a top-5 = {np.sort(ma)[::-1][:5].round(5)}")
print(f"identity: marg_b top-5 = {np.sort(mb)[::-1][:5].round(5)}")
print(f"identity: sum(marg_a) = {ma.sum():.6f}  (should be 1)")
print(f"alpha == beta marginals? {np.allclose(ma, mb)}  (expect True: closed shell)\n")

# --- Do different permutations actually give different marginals? ---
print("Marginal top-1 across 6 permutations:")
for i in range(6):
    perm = np.arange(NORB) if i == 0 else rng.permutation(NORB)
    op = apply_mask(permute_operator(op_full, perm))
    a, b = marginals_exact(op)
    print(f"  perm {i}: top1_a={a.max():.6f}  top1_b={b.max():.6f}  "
          f"n_a>1e-6={np.sum(a>1e-6):3d}")

# --- Sanity: full-space diagonalisation must return CASCI ---
E_full = subspace_energy_exact(np.ones(DIM_A), np.ones(DIM_B), budget=DIM_A)
print(f"\nFull-space energy   = {E_full:.8f}")
print(f"CASCI reference     = {E_CASCI:.8f}")
print(f"difference (mHa)    = {(E_full - E_CASCI)*1000:.4f}   (should be ~0)")

HF state: number of nonzero entries = 1
HF state: argmax (a_idx, b_idx) = (np.int64(0), np.int64(0))
expected (0, 0) for the HF determinant

identity: marg_a top-5 = [0.98572 0.00318 0.00243 0.00232 0.00203]
identity: marg_b top-5 = [0.98572 0.00318 0.00243 0.00232 0.00203]
identity: sum(marg_a) = 1.000000  (should be 1)
alpha == beta marginals? True  (expect True: closed shell)

Marginal top-1 across 6 permutations:
  perm 0: top1_a=0.985722  top1_b=0.985722  n_a>1e-6= 36
  perm 1: top1_a=0.991874  top1_b=0.991874  n_a>1e-6= 32
  perm 2: top1_a=0.995464  top1_b=0.995464  n_a>1e-6= 29
  perm 3: top1_a=0.987821  top1_b=0.987821  n_a>1e-6= 32
  perm 4: top1_a=0.993004  top1_b=0.993004  n_a>1e-6= 32
  perm 5: top1_a=0.992561  top1_b=0.992561  n_a>1e-6= 30

Full-space energy   = -108.84430043
CASCI reference     = -108.84430043
difference (mHa)    = -0.0000   (should be ~0)


In [8]:
from pyscf.fci import cistring

# Map ffsim alpha-string index -> bitstring in sbd convention
_strs = cistring.make_strings(range(NORB), NELEC[0])
def idx_to_bitstring(i):
    return format(_strs[i], f"0{NORB}b")

# ---- 1. Do exact and sampled selections agree? ----
bits = pd.read_csv(OUT / "seed_replication_n2_cas610_155.csv")
print("TEST 1: overlap between exact top-15 and sbd's sampled top-15\n")

for name in ["p13", "p07", "identity", "p23"]:
    perm = orderings[name]
    op = apply_mask(permute_operator(op_full, perm))
    ma, _ = marginals_exact(op)
    exact_set = {idx_to_bitstring(i) for i in np.argsort(ma)[::-1][:BUDGET]}

    f = Path.home() / "sqd-project/outputs/ordering" / f"{name}_adets.txt"
    if f.exists():
        sampled_set = set(f.read_text().split())
        ov = len(exact_set & sampled_set)
        print(f"  {name:9s} overlap {ov:2d}/{BUDGET}   "
              f"exact-only {len(exact_set-sampled_set)}, "
              f"sampled-only {len(sampled_set-exact_set)}")
    else:
        print(f"  {name:9s} (determinant file not found at {f})")

# ---- 2. Does agreement improve at larger budget? ----
print("\nTEST 2: fast-vs-sbd correlation as a function of budget\n")
sbd_mean = bits.groupby("ordering")["err_sub_mHa"].mean()

for B in [15, 25, 40, 60]:
    vals = []
    for name, perm in orderings.items():
        op = apply_mask(permute_operator(op_full, perm))
        ma, mb = marginals_exact(op)
        vals.append(subspace_energy_exact(ma, mb, budget=B))
    r = spearmanr(vals, [sbd_mean[n] for n in orderings]).statistic
    err = (np.array(vals) - E_CASCI) * 1000
    print(f"  budget {B:3d} (dim {B*B:5d}):  rho vs sbd = {r:+.3f}   "
          f"fast err range {err.min():6.2f} to {err.max():6.2f} mHa")

TEST 1: overlap between exact top-15 and sbd's sampled top-15

  p13       overlap 11/15   exact-only 4, sampled-only 4
  p07       overlap  9/15   exact-only 6, sampled-only 6
  identity  overlap 12/15   exact-only 3, sampled-only 3
  p23       overlap  8/15   exact-only 7, sampled-only 7

TEST 2: fast-vs-sbd correlation as a function of budget

  budget  15 (dim   225):  rho vs sbd = -0.083   fast err range  28.60 to 102.01 mHa
  budget  25 (dim   625):  rho vs sbd = +0.078   fast err range  11.33 to  39.38 mHa
  budget  40 (dim  1600):  rho vs sbd = -0.051   fast err range   3.36 to  16.69 mHa
  budget  60 (dim  3600):  rho vs sbd = -0.063   fast err range   1.05 to   7.49 mHa


In [9]:
from pyscf.fci import cistring

_strs = cistring.make_strings(range(NORB), NELEC[0])
bits_to_idx = {format(s, f"0{NORB}b"): i for i, s in enumerate(_strs)}

def energy_from_files(adet_path, bdet_path):
    """Diagonalise H in the product space defined by sbd's own determinant files."""
    ia = [bits_to_idx[b] for b in Path(adet_path).read_text().split()]
    ib = [bits_to_idx[b] for b in Path(bdet_path).read_text().split()]
    idx = np.array([a * DIM_B + b for a in ia for b in ib])
    n = len(idx)
    basis = np.zeros((DIM_A * DIM_B, n))
    basis[idx, np.arange(n)] = 1.0
    Hs = basis.T @ np.column_stack([ham @ basis[:, k] for k in range(n)])
    return float(np.linalg.eigvalsh((Hs + Hs.T) / 2)[0]), len(ia), len(ib)

ORD = Path.home() / "sqd-project/outputs/ordering"
print("Files available:", len(list(ORD.glob("*_adets.txt"))))
print(sorted(p.name for p in ORD.glob("*_adets.txt"))[:6], "...\n")

rep = pd.read_csv(OUT / "seed_replication_n2_cas610_155.csv")

print(f"{'file':22s} {'my E (mHa)':>12s} {'sbd E (mHa)':>12s} {'diff':>8s}  n_a n_b")
print("-" * 70)
for name in ["identity", "p13", "p07", "p23"]:
    for cand in [f"s2026_{name}", name]:
        a, b = ORD / f"{cand}_adets.txt", ORD / f"{cand}_bdets.txt"
        if a.exists() and b.exists():
            E, na, nb = energy_from_files(a, b)
            mine = (E - E_CASCI) * 1000
            row = rep[rep.ordering == name]
            theirs = row.err_sub_mHa.mean() if len(row) else float("nan")
            print(f"{cand:22s} {mine:12.3f} {theirs:12.3f} "
                  f"{mine-theirs:8.3f}  {na:3d} {nb:3d}")
            break
    else:
        print(f"{name:22s} (no files found)")

Files available: 159
['identity_adets.txt', 'p01_adets.txt', 'p02_adets.txt', 'p03_adets.txt', 'p04_adets.txt', 'p05_adets.txt'] ...

file                     my E (mHa)  sbd E (mHa)     diff  n_a n_b
----------------------------------------------------------------------
s2026_identity               43.040       26.018   17.022   15  15
s2026_p13                    32.430       19.978   12.452   15  15
s2026_p07                    69.999       20.261   49.739   15  15
s2026_p23                   145.806       96.580   49.225   15  15


In [10]:
# HF-only subspace: the answer is known exactly.
hf_a = ORD.parent / "n2_cas610_155_hf_only_adets.txt"
hf_b = ORD.parent / "n2_cas610_155_hf_only_bdets.txt"

if hf_a.exists():
    print("HF file contents:", repr(hf_a.read_text()))
    E, na, nb = energy_from_files(hf_a, hf_b)
    print(f"my HF-only energy = {E:.8f}")
    print(f"E_HF reference    = {E_HF:.8f}")
    print(f"difference (mHa)  = {(E - E_HF)*1000:.4f}   <- must be ~0")
else:
    print("not found; listing:", sorted(p.name for p in ORD.parent.glob("*hf_only*")))

# And check the mapping directly
hf_bits = "0" * (NORB - NELEC[0]) + "1" * NELEC[0]
print(f"\nHF bitstring '{hf_bits}' -> index {bits_to_idx.get(hf_bits)}   (expect 0)")

HF file contents: '0000000111\n'
my HF-only energy = -108.58732256
E_HF reference    = -108.58732256
difference (mHa)  = -0.0000   <- must be ~0

HF bitstring '0000000111' -> index 0   (expect 0)
